<a href="https://colab.research.google.com/github/EoniseLeannePMagno/Final-Project-CS2/blob/main/rosal_students_DB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import json
url = "https://raw.githubusercontent.com/EoniseLeannePMagno/Final-Project-CS2/refs/heads/main/data/students.json"
response = requests.get(url)
subjects = response.json()
print(subjects)

[{'id': 1, 'name': 'Maria Santos', 'section': '7-Ampere', 'subjects': {'Math2': 1.5, 'Math3': 1.75, 'Biology1': 2.0, 'Chem1': 1.25, 'Physics1': 1.75, 'Computer Science2': 1.25, 'PEHM2': 1.0, 'VE2': 1.75, 'Social Science2': 1.5, 'English2': 1.25, 'Filipino2': 1.75}}, {'id': 2, 'name': 'Juan Dela Cruz', 'section': '7-Curie', 'subjects': {'Math2': 2.5, 'Math3': 2.75, 'Biology1': 2.0, 'Chem1': 2.25, 'Physics1': 2.5, 'Computer Science2': 2.0, 'PEHM2': 1.75, 'VE2': 2.5, 'Social Science2': 2.25, 'English2': 2.0, 'Filipino2': 2.75}}, {'id': 3, 'name': 'Ana Reyes', 'section': '7-Rutherford', 'subjects': {'Math2': 1.0, 'Math3': 1.25, 'Biology1': 1.5, 'Chem1': 1.0, 'Physics1': 1.25, 'Computer Science2': 1.0, 'PEHM2': 1.25, 'VE2': 1.25, 'Social Science2': 1.0, 'English2': 1.25, 'Filipino2': 1.5}}, {'id': 4, 'name': 'Carlo Mendoza', 'section': '7-Newton', 'subjects': {'Math2': 2.75, 'Math3': 3.0, 'Biology1': 2.5, 'Chem1': 2.75, 'Physics1': 2.5, 'Computer Science2': 2.25, 'PEHM2': 2.0, 'VE2': 2.5, '

In [ ]:
!pip install firebase-admin


In [1]:
import firebase_admin
from firebase_admin import credentials, db
# Load the private key
cred = credentials.Certificate("/content/rosal-students-db-firebase-adminsdk-fbsvc-acdb682cd5.json")
# Initialize the app with your database URL
firebase_admin.initialize_app(cred, {
"databaseURL": "https://rosal-students-db-default-rtdb.asia-southeast1.firebasedatabase.app/"
})
print("Firebase connected successfully!")

Firebase connected successfully!


In [ ]:
import requests
import json
"""
The 'subjects' variable is already loaded from the URL in a previous cell.
Therefore, we can directly use the 'subjects' variable.
"""
data = subjects
print("JSON file loaded")
ref = db.reference("students")
for student in data:
  ref.child(str(student["id"])).set(student)
print("Data uploaded successfully!")

JSON file loaded
Data uploaded successfully!


In [ ]:
import firebase_admin
from firebase_admin import credentials, db

# Check if Firebase app is already initialized
try:
    firebase_admin.get_app()
except ValueError:
    # Firebase app not initialized, so initialize it
    cred = credentials.Certificate("/content/firebase_key.json")
    firebase_admin.initialize_app(cred, {
     "databaseURL": "https://rosal-students-db-default-rtdb.asia-southeast1.firebasedatabase.app/"
    })
    print("Firebase connected successfully (initialized in this cell)!")
else:
    print("Firebase app already initialized.")

student_name_to_search = input("Enter the name of the student to search: ")

ref = db.reference("students")
all_students = ref.get()

found = False

if all_students:
    for student_data in all_students:
        if student_data and "name" in student_data and student_name_to_search.lower() in student_data["name"].lower():
            print(f"\nRecord found for {student_name_to_search}:")
            print(f"  ID: {student_data['id']}")
            print(f"  Name: {student_data['name']}")
            print(f"  Section: {student_data['section']}")
            print("  Subjects:")
            if "subjects" in student_data:
                for subject, grade in student_data["subjects"].items():
                    print(f"    {subject}: {grade}")
            else:
                print("    No subjects recorded.")
            found = True
            break

if not found:
    print(f"\nRecord for '{student_name_to_search}' not found.")

Firebase app already initialized.
Enter the name of the student to search: Maria Santos

Record found for Maria Santos:
  ID: 1
  Name: Maria Santos
  Section: 7-Ampere
  Subjects:
    Biology1: 2.0
    Chem1: 1.25
    Computer Science2: 1.25
    English2: 1.25
    Filipino2: 1.75
    Math2: 1.5
    Math3: 1.75
    PEHM2: 1.0
    Physics1: 1.75
    Social Science2: 1.5
    VE2: 1.75


In [2]:
ref = db.reference("students")

while True:
    print("\n===== STUDENT DATABASE MENU ====")
    print("1. Display Students")
    print("2. Add Student")
    print("3. Update Student")
    print("4. Delete Student")
    print("5. Exit")

    choice = input("Enter choice: ")

    # DISPLAY
    if choice == "1":
        students_data = ref.get()
        print("\nStudent List:")
        if students_data:
            # Firebase Realtime Database can return a list when keys are sequential integers.
            # This list might contain None at index 0 if IDs start from 1.
            for student_item in students_data:
                if student_item: # Filter out None values
                    student_id = student_item.get('id', 'N/A')
                    student_name = student_item.get('name', 'N/A')
                    student_section = student_item.get('section', 'N/A')
                    print(f"ID: {student_id}, Name: {student_name}, Section: {student_section}")
                    # Display subjects if available
                    if 'subjects' in student_item:
                        print("  Subjects:")
                        for subject_name, grade_value in student_item['subjects'].items():
                            print(f"    {subject_name}: {grade_value}")
                    else:
                        print("  No subjects recorded.")
        else:
            print("No students found in the database.")

    # ADD
    elif choice == "2":
        sid = input("Enter ID: ")
        name = input("Enter name: ")
        section = input("Enter section: ")
        math = float(input("Enter Math2 grade: ")) # Using float for grades
        cs = float(input("Enter Computer Science2 grade: ")) # Using float for grades

        student = {
            "id": int(sid),
            "name": name,
            "section": section,
            "subjects": { # Changed to 'subjects' for consistency with existing data
                "Math2": math,
                "Computer Science2": cs
            }
        }

        ref.child(sid).set(student)
        print("Student added successfully!")

    # UPDATE
    elif choice == "3":
        sid = input("Enter ID of student to update: ")
        student_ref = ref.child(sid)

        student = student_ref.get()

        if student:
            name = input(f"Enter new name (current: {student.get('name', 'N/A')}): ")
            section = input(f"Enter new section (current: {student.get('section', 'N/A')}): ")
            # Retrieve current subject grades to display as default for update
            current_math_grade = student.get('subjects', {}).get('Math2', 'N/A')
            current_cs_grade = student.get('subjects', {}).get('Computer Science2', 'N/A')

            math = float(input(f"Enter new Math2 grade (current: {current_math_grade}): "))
            cs = float(input(f"Enter new Computer Science2 grade (current: {current_cs_grade}): "))

            student_ref.update({
                "name": name,
                "section": section,
                "subjects": {
                    "Math2": math,
                    "Computer Science2": cs
                }
            })

            print("Student updated successfully!")
        else:
            print("Student not found.")

    # DELETE
    elif choice == "4":
        sid = input("Enter ID to delete: ")

        if ref.child(sid).get():
            ref.child(sid).delete()
            print("Student deleted successfully!")
        else:
            print("Student not found.")

    # EXIT
    elif choice == "5":
        print("Exiting program...")
        break

    else:
        print("Invalid choice. Try again.")

"""
Loop → keeps menu running

Conditionals (if-elif-else) → handles choices
CRUD operations:
Create → set()
Read → get()
Update → update()
Delete → delete()
Dictionary (JSON) → data structure
Firebase interaction
"""


===== STUDENT DATABASE MENU ====
1. Display Students
2. Add Student
3. Update Student
4. Delete Student
5. Exit
Enter choice: 1

Student List:
ID: 1, Name: Maria Santos, Section: 7-Ampere
  Subjects:
    Biology1: 2.0
    Chem1: 1.25
    Computer Science2: 1.25
    English2: 1.25
    Filipino2: 1.75
    Math2: 1.5
    Math3: 1.75
    PEHM2: 1.0
    Physics1: 1.75
    Social Science2: 1.5
    VE2: 1.75
ID: 2, Name: Juan Dela Cruz, Section: 7-Curie
  Subjects:
    Biology1: 2.0
    Chem1: 2.25
    Computer Science2: 2.0
    English2: 2.0
    Filipino2: 2.75
    Math2: 2.5
    Math3: 2.75
    PEHM2: 1.75
    Physics1: 2.5
    Social Science2: 2.25
    VE2: 2.5
ID: 3, Name: Ana Reyes, Section: 7-Rutherford
  Subjects:
    Biology1: 1.5
    Chem1: 1.0
    Computer Science2: 1.0
    English2: 1.25
    Filipino2: 1.5
    Math2: 1.0
    Math3: 1.25
    PEHM2: 1.25
    Physics1: 1.25
    Social Science2: 1.0
    VE2: 1.25
ID: 4, Name: Carlo Mendoza, Section: 7-Newton
  Subjects:
    Biology1: 

'\nLoop → keeps menu running\n\nConditionals (if-elif-else) → handles choices\nCRUD operations:\nCreate → set()\nRead → get()\nUpdate → update()\nDelete → delete()\nDictionary (JSON) → data structure\nFirebase interaction\n'